In [1]:
!pip install transformers datasets scikit-learn accelerate -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving test_with_labels.csv to test_with_labels.csv
Saving trainV2.csv to trainV2.csv


In [3]:
import pandas as pd

train_df = pd.read_csv("trainV2.csv")
test_df = pd.read_csv("test_with_labels.csv")

def normalize_label(label):
    label = str(label).strip().lower()
    if label == "non-abusive":
        return 1
    else:
        return 0

train_df["Class"] = train_df["Class"].apply(normalize_label)
test_df["Class"] = test_df["Class"].apply(normalize_label)

train_df.head()

,Text,Class
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,1
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,0
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,1
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,0
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,1


In [4]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["Text"],
    train_df["Class"],
    test_size=0.1,
    stratify=train_df["Class"],
    random_state=42
)

In [5]:
from transformers import AutoTokenizer
from huggingface_hub import notebook_login

notebook_login()
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

In [6]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_df["Text"]), truncation=True, padding=True, max_length=128)

In [7]:
import torch

class TamilDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = TamilDataset(train_encodings, train_labels)
val_dataset = TamilDataset(val_encodings, val_labels)
test_dataset = TamilDataset(test_encodings, test_df["Class"])

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "ai4bharat/indic-bert",
    num_labels=2
)

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.weight       | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": round(acc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.665543,0.639300,0.702100,0.523800,0.600000
2,No log,0.608492,0.688500,0.711900,0.666700,0.688500
3,0.632823,0.582145,0.704900,0.690100,0.777800,0.731300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['albert.embeddings.LayerNorm.weight', 'albert.embeddings.LayerNorm.bias', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.weight', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['albert.embeddings.LayerNorm.beta', 'albert.embeddings.LayerNorm.gamma', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.beta', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.gamma'].


TrainOutput(global_step=618, training_loss=0.6123828023768552, metrics={'train_runtime': 233.5362, 'train_samples_per_second': 42.212, 'train_steps_per_second': 2.646, 'total_flos': 58896871787520.0, 'train_loss': 0.6123828023768552, 'epoch': 3.0})

In [14]:
test_results = trainer.evaluate(test_dataset)

print("\nFinal Test Results:")
print(f"Accuracy : {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall   : {test_results['eval_recall']:.4f}")
print(f"F1 Score : {test_results['eval_f1']:.4f}")


Final Test Results:
Accuracy : 0.7054
Precision: 0.6979
Recall   : 0.7585
F1 Score : 0.7269
